# openenv-dsc-co: grpo training on kaggle

end-to-end grpo run on kaggle t4x2 or p100. loads qwen2.5-coder-7b-instruct (4-bit), fine-tunes with trl grpo against the in-process dsc environment, and evaluates before / after on tier 1 and tier 2 scenarios.

**kaggle settings**: accelerator = `t4 x2` or `p100`; internet = on; persistence = variables & files.

if you hit vram pressure, flip `MODEL_NAME` below to `unsloth/Qwen2.5-Coder-3B-Instruct` or lower `NUM_GEN`.

## 1. fetch the env code
two paths. pick one and skip the other.

In [ ]:
import os, subprocess

HF_SPACE = os.environ.get('DSC_HF_SPACE', '')

if HF_SPACE:
    subprocess.check_call(['git', 'clone', f'https://huggingface.co/spaces/{HF_SPACE}', '/kaggle/working/dsc'])
else:
    subprocess.check_call(['git', 'clone', 'https://github.com/CYCLOP5/metascaler-hack', '/kaggle/working/dsc'])

os.chdir('/kaggle/working/dsc')
print('cwd', os.getcwd())
print('files', os.listdir('.'))

## 2. install training deps
unsloth + trl + pulp. **kaggle’s default image can ship a broken combo** (e.g. torch `2.11+cu130` + torchvision built for cu128, and bitsandbytes expecting `libnvJitLink.so.13` without cu13 libs on `LD_LIBRARY_PATH`). the install cell **realigns torch + torchvision + torchaudio + bitsandbytes on the official cu128 wheel index**, then installs unsloth **with `--no-deps`** so it cannot pull transformers 5.x.

**rules:**
- **do not** run bare `pip install trl` or `pip install unsloth` without `--no-deps` after pinning: resolver pulls trl 1.x + transformers 5.x and breaks `PatchFastRL` / unsloth patches.
- **do not** downgrade `numpy` to 1.x: torch builds here expect numpy 2.x.
- **`_center` / numpy ABI errors** are transient stale `.so` artefacts; the cell ends with a **kernel restart**.
- after restart, the verify cell prints versions — if `transformers` is not `4.46.x`, re-run section 2 only (do not skip the restart).

In [ ]:
%%capture
# Realign GPU stack on cu128 (see section 2 markdown): fixes torch/torchvision CUDA mismatch and
# bitsandbytes "libnvJitLink.so.13" when the image mixed cu130 torch with cu128 torchvision / no cu13 libs.
!pip install --force-reinstall --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install --upgrade --no-cache-dir bitsandbytes

# pydantic: needs deps so pydantic-core version matches
!pip install --no-deps --upgrade pulp fastapi 'uvicorn[standard]' fastmcp gymnasium
!pip install --upgrade pydantic

# Pinned HF stack first — unsloth MUST be --no-deps or pip upgrades transformers/trl via unsloth metadata.
!pip install --upgrade trl==0.29.0 transformers==4.46.3 accelerate==1.2.1 peft==0.13.2 datasets==3.2.0
!pip install --upgrade --no-deps unsloth==2025.9.5 unsloth_zoo==2025.9.12

!pip install --force-reinstall --no-cache-dir --no-deps transformers==4.46.3 'tokenizers>=0.20.3,<0.21'
!pip install --force-reinstall --no-cache-dir trl==0.29.0
!pip install --force-reinstall --no-cache-dir --no-deps unsloth_zoo==2025.9.12
!pip install --force-reinstall --no-cache-dir --no-deps transformers==4.46.3 'tokenizers>=0.20.3,<0.21'

!pip install --upgrade trackio
!apt-get install -y coinor-cbc 2>/dev/null || true

print('install done — restarting kernel to flush stale .so files...')
import IPython
IPython.get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import importlib.metadata as _im
import numpy as np
import torch, pulp, pydantic

print('torch      ', torch.__version__, '  cuda', torch.cuda.is_available(), 'gpus', torch.cuda.device_count())
print('numpy      ', np.__version__)
print('transformers', _im.version('transformers'), '  trl', _im.version('trl'))
print('pulp       ', pulp.__version__, '  pydantic', pydantic.VERSION)
try:
    print('unsloth_zoo', _im.version('unsloth_zoo'))
except _im.PackageNotFoundError:
    print('ERROR: unsloth_zoo MISSING — re-run install cell then restart kernel')

_tfm = _im.version('transformers')
if not _tfm.startswith('4.46.'):
    print('ERROR: need transformers 4.46.x for this Unsloth GRPO pin; got', _tfm, '— re-run section 2 install (full run + kernel restart).')
if not _im.version('trl').startswith('0.29.'):
    print('WARN: expected trl 0.29.x; got', _im.version('trl'))

## 3. sanity-check env before training

In [ ]:
from server.policies import zero_op_rollout, greedy_rollout, optimal_replay_rollout
import json

baselines = {}
for tier in [1, 2]:
    for name, fn in [('zero_op', zero_op_rollout), ('greedy', greedy_rollout), ('opt_replay', optimal_replay_rollout)]:
        r = fn(seed=7, difficulty=tier)
        baselines[f'tier{tier}/{name}'] = {'gap': r.gap, 'terminal': r.terminal, 'agent': r.agent_cost, 'opt': r.optimal_cost}
print(json.dumps(baselines, indent=2))

## 4. configure training run

In [ ]:
import os

MODEL_NAME = 'unsloth/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ = 8192
LORA_R = 32
DIFFICULTY = 1
NUM_GEN = 8
DATASET_SIZE = 500
MAX_STEPS = 120

os.environ['DSC_MODEL'] = MODEL_NAME
os.environ['DSC_MAX_SEQ'] = str(MAX_SEQ)
os.environ['DSC_LORA_R'] = str(LORA_R)
os.environ['DSC_TIER'] = str(DIFFICULTY)
os.environ['DSC_NUM_GEN'] = str(NUM_GEN)
os.environ['DSC_DATA_N'] = str(DATASET_SIZE)

print('config:', MODEL_NAME, 'tier=', DIFFICULTY, 'num_gen=', NUM_GEN, 'data_n=', DATASET_SIZE)

## 5. load model (unsloth 4-bit + vllm)

In [ ]:
import unsloth  # before other HF imports so patches apply (see unsloth warning)
from unsloth import FastLanguageModel, PatchFastRL

PatchFastRL(algorithm='grpo', FastLanguageModel=FastLanguageModel)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    fast_inference=True,
    gpu_memory_utilization=0.55,
    max_lora_rank=LORA_R,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_R,
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('model ready')

## 6. grpo training loop

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from train import DSCToolEnv, reward_func, schema_reward_func, SYSTEM_PROMPT

prompts = [[
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'plan the supply chain. call tools to minimize cost.'},
]] * DATASET_SIZE
dataset = Dataset.from_dict({'prompt': prompts})

config = GRPOConfig(
    output_dir='/kaggle/working/grpo_dsc_co',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    num_generations=NUM_GEN,
    max_completion_length=4096,
    beta=0.04,
    loss_type='bnpo',
    temperature=0.7,
    use_vllm=True,
    vllm_mode='colocate',
    chat_template_kwargs={'enable_thinking': False},
    log_completions=True,
    logging_steps=1,
    save_steps=40,
    max_steps=MAX_STEPS,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=[reward_func, schema_reward_func],
    environment_factory=DSCToolEnv,
)
trainer.train()

## 7. save lora + upload to hf hub

In [ ]:
OUT = '/kaggle/working/grpo_dsc_co/final'
trainer.save_model(OUT)
print('saved lora', OUT)

HF_REPO = os.environ.get('DSC_HF_REPO', '')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_REPO and HF_TOKEN:
    from huggingface_hub import HfApi
    HfApi().create_repo(HF_REPO, token=HF_TOKEN, repo_type='model', exist_ok=True)
    HfApi().upload_folder(folder_path=OUT, repo_id=HF_REPO, repo_type='model', token=HF_TOKEN)
    print('pushed', HF_REPO)

## 8. eval trained policy vs baselines

In [ ]:
import json, statistics
from train import DSCToolEnv
from server.dsc_environment import HORIZON

FastLanguageModel.for_inference(model)

def rollout_policy(seed, difficulty):
    envw = DSCToolEnv()
    os.environ['DSC_TIER'] = str(difficulty)
    envw.reset()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': 'plan the supply chain. call tools to minimize cost.'},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', tokenize=True, add_generation_prompt=True).to(model.device)
    out = model.generate(inputs, max_new_tokens=2048, temperature=0.3, do_sample=True)
    return {'seed': seed, 'difficulty': difficulty, 'cumulative': envw.cumulative, 'terminal': envw.terminal}

rewards = []
for s in range(5):
    rewards.append(rollout_policy(s, 1))
print('trained policy tier1:', json.dumps(rewards, indent=2))